<a href="https://colab.research.google.com/github/bangash-ds/flyrank-ml-internship/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bangash-ds/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

##**My Lane as an ML Task:**  My ML task is binary classification used to support ranking. The target is binary, so I can train a classifier to estimate the probability that a page satisfies the defined declining label. I can then sort pages by this probability and use the resulting ranked list to decide which pages should be reviewed first.

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Dataset loaded:", df.shape)

Dataset loaded: (30000, 45)


In [ ]:
df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

- My target is is_declining_label, a binary label defined by the data pipeline as 1 when trend_direction is down and 0 otherwise. It is a defined rule based on the change in impressions between the most recent 30 days and the preceding 30 days, rather than an independently observed content-refresh outcome. Therefore, the model should be interpreted as identifying pages that satisfy this decline definition, not as predicting that a refresh will improve performance.

In [ ]:
print(df["trend_direction"].value_counts(dropna=False))

print(df["is_declining_label"].value_counts())

display(
    pd.crosstab(
        df["trend_direction"],
        df["is_declining_label"]
    )
)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


is_declining_label,0,1
trend_direction,,
down,0,16262
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

- I will use Precision@K because the goal is to prioritize a limited number of pages for review. A good model should place a high proportion of pages satisfying the defined decline label near the top of the ranked queue. K should represent a realistic review capacity rather than being chosen arbitrarily.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nUnique content IDs:", df["content_id"].nunique())

print("Total rows:", len(df))

print("Unique clients:", df["client_id"].nunique())

Shape: (30000, 45)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



Unique content IDs: 30000
Total rows: 30000
Unique clients: 32


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
- A fixed rule may be too limited because page-prioritization can depend on multiple search, content, keyword, and engagement signals rather than one threshold. A classification model can combine these signals into a probability score that can be used to rank pages. However, whether ML actually improves on a fixed rule must be tested against a baseline rather than assumed.

In [ ]:
signal_columns = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate"
]

display(df[signal_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
pageviews_90d,30000.0,49.942467,152.101430,0.0,2.0,8.00,33.00,5998.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
engagement_rate,30000.0,2.534520,8.310096,0.0,0.0,0.00,1.35,100.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.